# TenderLens — Exploratory Data Analysis

**Дата:** 25.04.2026  
**Автор:** TenderLens Team  
**Цель:** Первичный анализ данных о госзакупках

---

## Содержание

1. Загрузка данных
2. Базовая статистика
3. Анализ цен
4. Анализ по регионам
5. Анализ по законам (44-ФЗ vs 223-ФЗ)
6. Топ заказчиков
7. Концентрация рынка
8. Визуализация

## 1. Загрузка данных

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
import glob
import sys

# Добавляем путь к модулям проекта
sys.path.append('..')
from analytics import pricing, competition

# Настройки pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Библиотеки загружены успешно!")

In [ ]:
# Загружаем последний файл с данными
data_files = sorted(glob.glob('../data/lots_all_*.json'))
latest_file = data_files[-1]

print(f"Загружаем данные из: {latest_file}")

with open(latest_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)

print(f"\nЗагружено {len(df)} лотов")
print(f"Колонки: {', '.join(df.columns)}")

## 2. Базовая статистика

In [ ]:
# Общая информация
print("=== ОБЩАЯ ИНФОРМАЦИЯ ===")
print(f"Всего лотов: {len(df):,}")
print(f"Уникальных заказчиков: {df['customer_name'].nunique():,}")
print(f"Регионов: {df['region_name'].nunique()}")
print(f"\nПериод данных: {df['scraped_at'].min()} — {df['scraped_at'].max()}")

# Информация о типах данных
print("\n=== ТИПЫ ДАННЫХ ===")
df.info()

In [ ]:
# Проверка пропусков
print("=== ПРОПУЩЕННЫЕ ЗНАЧЕНИЯ ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Пропусков': missing,
    'Процент': missing_pct
})
print(missing_df[missing_df['Пропусков'] > 0])

if missing_df['Пропусков'].sum() == 0:
    print("\nПропусков нет!")

## 3. Анализ цен

In [ ]:
# Используем модуль pricing
print(pricing.price_summary(df))

In [ ]:
# Статистика цен по регионам
print("=== ЦЕНЫ ПО РЕГИОНАМ ===")
price_by_region = pricing.price_stats_by_region(df)
price_by_region

In [ ]:
# Статистика цен по законам
print("=== ЦЕНЫ ПО ЗАКОНАМ ===")
price_by_law = pricing.price_stats_by_law(df)
price_by_law

In [ ]:
# Выявление выбросов
normal, outliers = pricing.identify_outliers(df, method='iqr')

print(f"Нормальных лотов: {len(normal):,}")
print(f"Выбросов: {len(outliers):,} ({len(outliers)/len(df)*100:.2f}%)")

if len(outliers) > 0:
    print("\nПримеры выбросов (топ-5 по цене):")
    print(outliers.nlargest(5, 'initial_price')[['reg_number', 'object_name', 'initial_price', 'customer_name']])

## 4. Анализ по регионам

In [ ]:
# Используем модуль competition
region_stats = competition.analyze_by_region(df)
region_stats

## 5. Анализ по законам (44-ФЗ vs 223-ФЗ)

In [ ]:
law_stats = competition.analyze_by_law(df)
law_stats

## 6. Топ заказчиков

In [ ]:
# Топ-10 по количеству лотов
print("=== ТОП-10 ЗАКАЗЧИКОВ ПО КОЛИЧЕСТВУ ЛОТОВ ===")
top_by_count = competition.top_customers(df, n=10, by='count')
top_by_count

In [ ]:
# Топ-10 по объёму закупок
print("=== ТОП-10 ЗАКАЗЧИКОВ ПО ОБЪЁМУ ЗАКУПОК ===")
top_by_volume = competition.top_customers(df, n=10, by='volume')
top_by_volume

## 7. Концентрация рынка

In [ ]:
# Анализ концентрации
print(competition.competition_summary(df))

In [ ]:
# Концентрация по объёму
conc_volume = competition.market_concentration(df, by='volume')
print("Концентрация по объёму:")
for key, value in conc_volume.items():
    print(f"  {key}: {value}")

## 8. Визуализация

In [ ]:
# Распределение цен (гистограмма)
fig = px.histogram(
    df, 
    x='initial_price',
    nbins=50,
    title='Распределение начальных цен закупок',
    labels={'initial_price': 'Начальная цена (руб.)', 'count': 'Количество лотов'}
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Распределение цен (box plot)
fig = px.box(
    df,
    y='initial_price',
    title='Box Plot: Распределение начальных цен',
    labels={'initial_price': 'Начальная цена (руб.)'}
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Распределение по регионам
region_counts = df['region_name'].value_counts()

fig = px.bar(
    x=region_counts.index,
    y=region_counts.values,
    title='Количество лотов по регионам',
    labels={'x': 'Регион', 'y': 'Количество лотов'}
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Распределение по законам (pie chart)
law_counts = df['law'].value_counts()

fig = px.pie(
    values=law_counts.values,
    names=law_counts.index,
    title='Распределение закупок по законам',
    hole=0.3
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Топ-10 заказчиков по объёму
top10 = competition.top_customers(df, n=10, by='volume')

fig = px.bar(
    x=top10['total_volume'],
    y=top10.index,
    orientation='h',
    title='Топ-10 заказчиков по объёму закупок',
    labels={'x': 'Общий объём (руб.)', 'y': 'Заказчик'}
)
fig.update_layout(height=600, yaxis={'categoryorder': 'total ascending'})
fig.show()

In [ ]:
# Сравнение цен по регионам (box plot)
fig = px.box(
    df,
    x='region_name',
    y='initial_price',
    title='Распределение цен по регионам',
    labels={'region_name': 'Регион', 'initial_price': 'Начальная цена (руб.)'}
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Сравнение цен по законам (box plot)
fig = px.box(
    df,
    x='law',
    y='initial_price',
    title='Распределение цен по законам (44-ФЗ vs 223-ФЗ)',
    labels={'law': 'Закон', 'initial_price': 'Начальная цена (руб.)'}
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Распределение по статусам
status_counts = df['status'].value_counts().head(10)

fig = px.bar(
    x=status_counts.values,
    y=status_counts.index,
    orientation='h',
    title='Топ-10 статусов закупок',
    labels={'x': 'Количество лотов', 'y': 'Статус'}
)
fig.update_layout(height=500, yaxis={'categoryorder': 'total ascending'})
fig.show()

## Выводы

1. **Объём данных:** Собрано 600 уникальных лотов из 3 регионов
2. **Качество данных:** Пропусков нет, все обязательные поля заполнены
3. **Распределение по законам:** 44-ФЗ доминирует (~90%), 223-ФЗ ~10%
4. **Цены:** Широкий диапазон от нескольких тысяч до сотен миллионов рублей
5. **Концентрация:** Рынок умеренно концентрирован, топ-10 заказчиков занимают значительную долю

---

**Следующие шаги:**
- Собрать больше данных (1000+ лотов)
- Добавить парсинг дат публикации и дедлайнов
- Реализовать расчёт снижения цены (когда появятся данные о ценах победителей)
- Построить модель скоринга ниш